# mu-logsigma-encoder-head — worked example 2: Implement a VAE encoder head as an nn.Module with double-width linear

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `mu-logsigma-encoder-head`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Packaging the VAE encoder head as an `nn.Module` makes it composable and ensures the learnable projection weights are registered as parameters. The standard construction uses a single `nn.Linear(d_in, 2 * latent_dim)` layer whose output is split into `mu` and `logsigma` inside `forward`. This is the form you will most often see in production VAE code.

## Worked solution

**Step 1 — declare the module.** Subclass `nn.Module`, call `super().__init__()`, and create `self.proj = nn.Linear(d_in, 2 * latent_dim)`. The doubled output dimension is what distinguishes a VAE encoder head from a plain projection layer.

**Step 2 — implement forward.** Call `self.proj(h)` to get `(B, 2 * latent_dim)`, then split with `chunk(2, dim=-1)`. This returns two tensors each of shape `(B, latent_dim)`.

**Step 3 — verify shapes.** Check that `mu.shape == (B, latent_dim)` and `logsigma.shape == (B, latent_dim)` with an assertion.

**Step 4 — verify parameter count.** The module should have `d_in * 2 * latent_dim + 2 * latent_dim` parameters (weight + bias of the single linear layer). Summing `p.numel()` over `.parameters()` confirms this.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

class VAEEncoderHead(nn.Module):
    def __init__(self, d_in: int, latent_dim: int):
        super().__init__()
        # Single linear that produces BOTH mu and logsigma concatenated
        self.proj = nn.Linear(d_in, 2 * latent_dim)
        self.latent_dim = latent_dim

    def forward(self, h: torch.Tensor):
        """h: (B, d_in) -> (mu, logsigma) each (B, latent_dim)"""
        params = self.proj(h)              # (B, 2 * latent_dim)
        mu, logsigma = params.chunk(2, dim=-1)
        return mu, logsigma

# Exercise: instantiate and verify shapes + parameter count.
torch.manual_seed(9)
d_in, latent_dim, B = 64, 16, 8
head = VAEEncoderHead(d_in, latent_dim)

h = torch.randn(B, d_in)
mu, logsigma = head(h)

print(f"mu shape:       {mu.shape}")
print(f"logsigma shape: {logsigma.shape}")
assert mu.shape == (B, latent_dim), f"Expected ({B}, {latent_dim}), got {mu.shape}"
assert logsigma.shape == (B, latent_dim)

n_params = sum(p.numel() for p in head.parameters())
expected_params = d_in * 2 * latent_dim + 2 * latent_dim
print(f"Parameters: {n_params} (expected {expected_params})")
assert n_params == expected_params
print("All assertions passed!")